In [1]:
# 09/14
from pathlib import Path
import torch

Path('models').mkdir(exist_ok=True)
print(torch.__version__)

2.7.1+cu128


In [2]:
print(torch.cuda.is_available())

True


In [3]:
print(torch.tensor([1,2,3]))
print(torch.Tensor([[1,2,3], [4,5,6]]))
print(torch.LongTensor([1,2,3]))
print(torch.FloatTensor([1,2,3]))

tensor([1, 2, 3])
tensor([[1., 2., 3.],
        [4., 5., 6.]])
tensor([1, 2, 3])
tensor([1., 2., 3.])


In [4]:
tensor = torch.rand(1, 2)
print(tensor.shape)
print(tensor.dtype)
print(tensor.device)

torch.Size([1, 2])
torch.float32
cpu


In [5]:
tensor = torch.rand((3,3), dtype = torch.float)
print(tensor)

tensor([[0.1412, 0.0216, 0.3929],
        [0.2280, 0.4030, 0.3248],
        [0.9691, 0.8203, 0.6299]])


In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [7]:
cpu = torch.tensor([1, 2, 3], dtype=torch.float32)
gpu = cpu.to(device)
tensor = torch.rand((1, 1), device=device)
print(cpu)
print(gpu)
print(tensor)

tensor([1., 2., 3.])
tensor([1., 2., 3.], device='cuda:0')
tensor([[0.6024]], device='cuda:0')


In [8]:
cpu = torch.tensor([1, 2, 3], dtype=torch.float32)
gpu = cpu.to(device)
gpu2cpu = gpu.cpu()
cpu2gpu = cpu.to(device)

In [9]:
import numpy as np

ndarray = np.array([1,2,3], dtype=np.uint8)
print(torch.tensor(ndarray))
print(torch.Tensor(ndarray))
print(torch.from_numpy(ndarray))

tensor([1, 2, 3], dtype=torch.uint8)
tensor([1., 2., 3.])
tensor([1, 2, 3], dtype=torch.uint8)


In [10]:
tensor = torch.tensor([1, 2, 3], dtype=torch.float32, device=device)
ndarray = tensor.detach().cpu().numpy()
print(ndarray)
print(tensor)

[1. 2. 3.]
tensor([1., 2., 3.], device='cuda:0')


In [11]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [12]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x = df.iloc[:,0].values
        self.y = df.iloc[:,1].values
        self.length = len(df)
    def __getitem__(self, index):
        x = torch.FloatTensor([self.x[index] ** 2, self.x[index]])
        y = torch.FloatTensor([self.y[index]])
        return x, y
    def __len__(self):
        return self.length

In [13]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(2,1)
    def forward(self, x):
        x = self.layer(x)
        return x

In [14]:
train_dataset = CustomDataset('./dataset/non_linear.csv')
train_dataloader = DataLoader(train_dataset,
                             batch_size=128,
                             shuffle=True,
                             drop_last=True)

In [15]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CustomModel().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.0001)

In [16]:
model.train()
for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cost += loss.item()
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 0:
        print(epoch +1, cost)

100 15.724306106567383
200 4.351937294006348
300 1.2290945053100586
400 0.5120731592178345
500 0.29668012261390686
600 0.23717036843299866
700 0.20706288516521454
800 0.1837007701396942
900 0.19773508608341217
1000 0.22570228576660156


In [17]:
with torch.no_grad():
    model.eval()
    inputs = torch.FloatTensor([[1**2, 1],
                                [5**2, 5],
                                [11**2, 11]]).to(device)
    outputs = model(inputs)
    print(outputs)

tensor([[  1.3730],
        [ 69.1731],
        [357.3835]], device='cuda:0')


In [18]:
# 모델 구조는 CustomModel로 재생성하고 가중치만 저장합니다.
torch.save(model.state_dict(), './models/model.pt')

In [19]:
torch.save(model.state_dict(), './models/model_state_dict.pt')

In [20]:
dataset = CustomDataset('./dataset/non_linear.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset,
                                                        [train_size,
                                                         val_size,
                                                         test_size])
train_dataloader = DataLoader(train_dataset,
                              batch_size=16,
                              shuffle=True,
                              drop_last=False)
val_dataloader = DataLoader(val_dataset,
                              batch_size=4,
                              shuffle=True,
                              drop_last=False)
test_dataloader = DataLoader(test_dataset,
                              batch_size=4,
                              shuffle=False,
                              drop_last=False)

In [21]:
with torch.no_grad():
    model.eval()
    val_loss = 0.0
    for x, y in val_dataloader:
        x = x.to(device)
        y = y.to(device)
        outputs = model(x)
        val_loss += criterion(outputs, y).item() * len(x)
    print('Validation MSE:', val_loss / len(val_dataloader.dataset))

Validation MSE: 0.09687249138951301


In [22]:
# 모델 구조는 CustomModel로 재생성하고 가중치만 저장합니다.
torch.save(model.state_dict(), './models/model.pt')

In [23]:
torch.save(model.state_dict(), './models/model_state_dict.pt')

In [24]:
model = CustomModel().to(device)
model.load_state_dict(torch.load('./models/model.pt', map_location=device, weights_only=True))
# 새 모델의 파라미터를 학습하도록 optimizer도 다시 연결합니다.
optimizer = optim.SGD(model.parameters(), lr=0.0001)

In [25]:
print(model)

CustomModel(
  (layer): Linear(in_features=2, out_features=1, bias=True)
)


In [26]:
model_state_dict = torch.load('./models/model_state_dict.pt', map_location=device, weights_only=True)
model.load_state_dict(model_state_dict)

<All keys matched successfully>

In [27]:
checkpoint = 1

model.train()
for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cost += loss.item()
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 0:
        print(epoch +1, cost)
        torch.save(model.state_dict(), f'./models/checkpoint-{checkpoint}.pt')
        checkpoint += 1

100 0.19069007262587548
200 0.17355345487594603
300 0.15397225618362426
400 0.14051381051540374
500 0.1304866787046194
600 0.12041178122162818
700 0.11246568895876408
800 0.10565769299864769
900 0.10071457996964454
1000 0.09684655666351319


In [28]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [29]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x1 = df.iloc[:,0].values
        self.x2 = df.iloc[:,1].values
        self.x3 = df.iloc[:,2].values
        self.y = df.iloc[:,3].values
        self.length = len(df)
    def __getitem__(self, index):
        x = torch.FloatTensor([self.x1[index],
                               self.x2[index],
                               self.x3[index]])
        y = torch.FloatTensor([int(self.y[index])])
        return x, y
    def __len__(self):
        return self.length

In [30]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(nn.Linear(3,1),
                                   nn.Sigmoid())

    def forward(self, x):
        x = self.layer(x)
        return x

In [31]:
dataset = CustomDataset('./dataset/binary.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size-train_size-val_size

train_dataset, val_dataset, test_dataset = random_split(dataset,
                                                        [train_size,
                                                         val_size,
                                                         test_size],
                                                         torch.manual_seed(42))
train_dataloader = DataLoader(train_dataset,
                              batch_size=64,
                              shuffle=True,
                              drop_last=False)
val_dataloader = DataLoader(val_dataset,
                              batch_size=4,
                              shuffle=True,
                              drop_last=False)
test_dataloader = DataLoader(test_dataset,
                              batch_size=4,
                              shuffle=False,
                              drop_last=False)

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomModel().to(device)
criterion = nn.BCELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr = 0.0001)

In [33]:
model.train()
for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cost += loss.item()
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 0:
        print(epoch + 1, cost)

100 0.6312185571743891
200 0.6317871304658743
300 0.6283998626929063
400 0.6280962870671198
500 0.6246365308761597
600 0.6282455416826102
700 0.6211693195196298
800 0.6250638182346637
900 0.6223363371995779
1000 0.6192680689004751


In [34]:
with torch.no_grad():
    model.eval()
    for x, y in val_dataloader:
        x = x.to(device)
        y = y.to(device)
        outputs = model(x)
        print(outputs)
        print(outputs >= torch.FloatTensor([0.5]).to(device))

tensor([[0.5799],
        [0.6213],
        [0.5699],
        [0.6135]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[0.5204],
        [0.5770],
        [0.5113],
        [0.4708]], device='cuda:0')
tensor([[ True],
        [ True],
        [ True],
        [False]], device='cuda:0')
tensor([[0.6815],
        [0.5921],
        [0.6303],
        [0.5596]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[0.6319],
        [0.6782],
        [0.6350],
        [0.4755]], device='cuda:0')
tensor([[ True],
        [ True],
        [ True],
        [False]], device='cuda:0')
tensor([[0.5783],
        [0.5870],
        [0.4960],
        [0.5519]], device='cuda:0')
tensor([[ True],
        [ True],
        [False],
        [ True]], device='cuda:0')
tensor([[0.6263],
        [0.5947],
        [0.5303],
        [0.6877]], device='cuda:0')
tensor([[True],
        [True],
      

In [35]:
from torch import nn


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(1, 2),
            nn.Sigmoid()
        )
        self.fc = nn.Linear(2, 1)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.layer[0].weight)
        self.layer[0].bias.data.fill_(0.01)

        nn.init.xavier_uniform_(self.fc.weight)
        self.fc.bias.data.fill_(0.01)


model = Net()

In [36]:
for name, param in model.named_parameters():
    print(name, param.data)

layer.0.weight tensor([[-0.2196],
        [ 1.1053]])
layer.0.bias tensor([0.0100, 0.0100])
fc.weight tensor([[ 0.9165, -0.2847]])
fc.bias tensor([0.0100])


In [37]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(1, 2),
            nn.Sigmoid()
        )
        self.fc = nn.Linear(2, 1)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.constant_(module.bias, 0.01)
        print(module)


model = Net().to(device)

Linear(in_features=1, out_features=2, bias=True)
Sigmoid()
Sequential(
  (0): Linear(in_features=1, out_features=2, bias=True)
  (1): Sigmoid()
)
Linear(in_features=2, out_features=1, bias=True)
Net(
  (layer): Sequential(
    (0): Linear(in_features=1, out_features=2, bias=True)
    (1): Sigmoid()
  )
  (fc): Linear(in_features=2, out_features=1, bias=True)
)


In [38]:
# model = nn.Linear(1, 1).to(device)
# optimizer = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=0.01)

In [39]:
import nlpaug.augmenter.word as naw


texts = [
    "Those who can imagine anything, can create the impossible.",
    "We can only see a short distance ahead, but we can see plenty there that needs to be done.",
    "If a machine is expected to be infallible, it cannot also be intelligent.",
]

aug = naw.ContextualWordEmbsAug(model_path="bert-base-uncased", action="insert")
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

c:\Users\KDS23\Documents\17-pytorch\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Those who can imagine anything, can create the impossible.
only those who do can imagine practically anything, can create the natural impossible.
We can only see a short distance ahead, but we can see plenty there that needs to be done.
here we can only now see a fairly short distance straight ahead, but we sure can see plenty there for that needs to be carefully done.
If a machine is expected to be infallible, it cannot also be intelligent.
if the a machine it is expected to also be approximately infallible, yet it cannot also technically be intelligent.


In [40]:
import nlpaug.augmenter.char as nac


texts = [
    "Those who can imagine anything, can create the impossible.",
    "We can only see a short distance ahead, but we can see plenty there that needs to be done.",
    "If a machine is expected to be infallible, it cannot also be intelligent.",
]

aug = nac.RandomCharAug(action="delete")
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
ose who can iane anything, can ceat the impoibe.
We can only see a short distance ahead, but we can see plenty there that needs to be done.
We can ly see a hot itnce aed, but we can see plenty thr at needs to be ne.
If a machine is expected to be infallible, it cannot also be intelligent.
If a chie is xpced to be inalibe, it cann ao be intelligent.


In [41]:
import nlpaug.augmenter.word as naw


texts = [
    "Those who can imagine anything, can create the impossible.",
    "We can only see a short distance ahead, but we can see plenty there that needs to be done.",
    "If a machine is expected to be infallible, it cannot also be intelligent.",
]

aug = naw.RandomWordAug(action="swap")
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Who those can imagine anything can, create the impossible.
We can only see a short distance ahead, but we can see plenty there that needs to be done.
We only can see distance a short ahead, we but can see there plenty needs that to be. done
If a machine is expected to be infallible, it cannot also be intelligent.
A if machine expected is to be, infallible cannot it also intelligent be.


In [42]:
import nlpaug.augmenter.word as naw


texts = [
    "Those who can imagine anything, can create the impossible.",
    "We can only see a short distance ahead, but we can see plenty there that needs to be done.",
    "If a machine is expected to be infallible, it cannot also be intelligent.",
]

aug = naw.SynonymAug(aug_src='wordnet')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Those world health organization lavatory imagine anything, prat create the unacceptable.
We can only see a short distance ahead, but we can see plenty there that needs to be done.
We lavatory only insure a short distance beforehand, merely we can fancy plenty on that point that needs to be done.
If a machine is expected to be infallible, it cannot also be intelligent.
If a automobile is expect to be infallible, information technology cannot besides be sound.


In [43]:

import nlpaug.augmenter.word as naw


texts = [
    "Those who can imagine anything, can create the impossible.",
    "We can only see a short distance ahead, but we can see plenty there that needs to be done.",
    "If a machine is expected to be infallible, it cannot also be intelligent.",
]
reserved_tokens = [
    ["can", "can't", "cannot", "could"],
]

reserved_aug = naw.ReservedAug(reserved_tokens=reserved_tokens)
augmented_texts = reserved_aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Those who could imagine anything, can't create the impossible.
We can only see a short distance ahead, but we can see plenty there that needs to be done.
We cannot only see a short distance ahead, but we cannot see plenty there that needs to be done.
If a machine is expected to be infallible, it cannot also be intelligent.
If a machine is expected to be infallible, it can't also be intelligent.


In [45]:
import nlpaug.augmenter.word as naw


texts = [
    "Those who can imagine anything, can create the impossible.",
    "We can only see a short distance ahead, but we can see plenty there that needs to be done.",
    "If a machine is expected to be infallible, it cannot also be intelligent.",
]

back_translation = naw.BackTranslationAug(
    from_model_name='facebook/wmt19-en-de', 
    to_model_name='facebook/wmt19-de-en',
    device = 'cpu'
)
augmented_texts = back_translation.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Anyone who can imagine anything can achieve the impossible.
We can only see a short distance ahead, but we can see plenty there that needs to be done.
We can only look a little ahead, but we can see a lot there that needs to be done.
If a machine is expected to be infallible, it cannot also be intelligent.
If a machine is expected to be infallible, it cannot be intelligent.
